---
format:
  html:
    code-fold: true
    code-summary: "Mostrar código"
---

# Introducción a los rendimientos queseros

La elaboración de queso tiene como objetivo principal **preservar y concentrar los componentes nutritivos de la leche**, transformándolos en un producto alimentario con **elevado valor sensorial, nutricional y comercial**. Este proceso da lugar a una amplia variedad de quesos, cada uno con características únicas que responden a factores como el tipo de leche, la tecnología aplicada y las condiciones de maduración.

El propósito de la tecnología quesera es **optimizar esta transformación**, guiando el proceso de forma que se obtenga el mejor producto posible, **garantizando la calidad sanitaria, tecnológica y organoléptica**, y todo ello de manera **eficiente y rentable**. Para lograrlo, los objetivos técnicos se centran en **minimizar las pérdidas de materia**, especialmente de **grasa y proteína**, que son los componentes clave en la formación del queso. Al mismo tiempo, se busca **preservar los atributos sensoriales** —como sabor, textura y aroma— y **cumplir con los requisitos higiénico-sanitarios** establecidos por la normativa vigente.

El análisis del proceso quesero se estructura en dos partes diferenciadas:

-   **Balance de materia global:** comprende desde la recepción de la leche cruda hasta el final del proceso de elaboración, momento en que el queso está listo para entrar en la fase de maduración. Este balance permite evaluar el rendimiento general y detectar posibles pérdidas o desviaciones.
-   **Transformación quesera propiamente dicha:** incluído en el balance de materia global, abarca el tramo desde la leche destinada a fabricación una vez en la cuba, hasta la obtención del queso fresco (antes o después del salado). Aquí se analizan los rendimientos específicos, la eficiencia de coagulación, el corte, el desuerado, el moldeado y otros parámetros tecnológicos que inciden directamente en la calidad y cantidad del producto final.

En este capítulo abordaremos la segunda parte, la transformación quesera, analizando los valores analíticos de la leche de fabricación y del queso recién elaborado, y extraeremos diversas consecuencias:

-   Estableceremos los parámetros de referencia de nuestro propio proceso, que nos servirán de punto de comparación para las fabricaciones que vayamos realizando.
-   Evaluaremos las causas posibles de las desviaciones, y veremos cómo proponer planes de acción para la reducción o eliminación de estas desviaciones.


## Un ejemplo de cálculo de rendimientos

En este ejemplo, vamos a suponer que elaboramos un queso de vaca, del que tenemos valores analíticos de producto terminado y de la leche utilizada en su elaboración. El proceso siempre tiene los mismos pasos tecnológicos, con la variabilidad que suponemos que es la habitual. 

Podemos imaginar dos situaciones en las que nos encontremos con la necesidad de analizar unos datos acumulados:

- Llegamos a una empresa en la que han estado acumulando datos de producción, pero no han sabido cómo formalizar y estructurar los datos adecuadamente
- Estamos trabajando en una empresa en la que se analizan los datos, y se plantean establecer un presupuesto más formal para el siguiente año, lo que exige el análisis de los datos del año en curso y el establecimiento de las hipótesis de presupuesto, entre ellas, el consumo de materia, que determinará las necesidades de aprovisionamiento y el precio de venta del queso.

En ambos casos, necesitaremos analizar un conjunto de datos de fabricación; siguiendo nuestra línea de trabajo, lo haremos con `python`y Excel.

Los datos técnicos de ejemplo que vamos a utilizar están registrados en un fichero de datos en formato `csv`, que incluye:

1.  las analíticas del queso, y
2.  las analíticas de la leche con la que lo hacemos.

Caracterizaremos nuestro queso mediante un análisis de extracto seco total (EST) y una grasa butirométrica (MG), en un punto determinado de nuestro proceso (por ejemplo, a la salida de la salmuera, después de escurrido) y mediante un muestreo repetible, de forma que nuestro análisis sea representativo. A partir de estos dos análisis, calculamos otros parámetros técnicos necesarios, tales como la materia grasa sobre el extracto seco total (G/ES), el extracto seco magro o desnatado (ESM) y la humedad del queso desnatado (HQD), valores que se calculan a partir de los primeros.

En el recorrido de cálculo utilizaremos `Python`, al final se proporciona una hoja de cálculo con la información necesaria.

Como siempre, es opcional descargar los datos de ejemplo y abrir el cuaderno en Colab para ejecutar el código a medida que se avanza en el estudio.

[Abrir este cuaderno en Google Colab](https://colab.research.google.com/github/juanriera/master-queseria/blob/master/085-intro-rendim.ipynb){target="_blank" rel="noopener noreferrer"}

[Descargar los datos de ejemplo utilizados en este cuaderno (archivo `fab_queso_bm.csv`)](https://raw.githubusercontent.com/juanriera/master-queseria/master/datos/fab_queso_bm_vaca.csv){target="_blank" rel="noopener noreferrer"}

## Lectura y exploración de los datos

Antes de lanzarnos al análisis cuantitativo, siempre es conveniente una primera visualización de los datos, por si hubiese algún valor que nos pueda resultar cuestionable. 

Es una mala práctica, aunque muy extendida, usar la hoja de cálculo para hacer la media aritmética de cada parámetro que vayamos a utilizar y trabajar con estos valores medios. En un conjunto de datos que recoja muchas fabricaciones, nunca podemos estar seguros de que el 100% de los valores sean correctos, que no haya habido errores de muestreo, analíticos o problemas de fabricación que hayan producido valores anormales. Por eso vamos a utilizar siempre los valores *medianos* y no los valores medios; como sabemos, la **mediana** es resistente a los valores anormales y extremos, mientras que la media no lo es.

Procedemos a leer los datos, y visualizamos las primeras lineas. Se incluye el código `python`que permite visualizar las tablas, en algunos casos se han formateado mediante `HTML`.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

url_datos = 'https://raw.githubusercontent.com/juanriera/master-queseria/master/datos/fab_queso_bm_vaca.csv'

try:
    df = pd.read_csv(url_datos, decimal = ",", sep=';', encoding='ISO-8859-1')
except Exception as e:
    print(f"Error al cargar el archivo: {e}")

# crear index con fecha
df['fecha_index'] = pd.to_datetime(
    df['fecha'], 
    format='%d/%m/%Y',  # Formato Día/Mes/Año confirmado
    errors='coerce'     # Clave: convierte los valores que no puede leer a NaT (Not a Time)
)
# 2. Limpiar NaT y establecer el índice
df.dropna(subset=['fecha_index'], inplace=True)
df.set_index('fecha_index', inplace=True)
df.sort_index(inplace=True)

df.head()

,fecha,receta,litros_past,est_past,mg_past,mp_past,kg_cuba,est_cuba,mg_cuba,mp_cuba,lactosa_cuba,ph_final_moldeo_cuba,formato,est_salida_salmuera,mg_salida_salmuera,ph_salida_salmuera,peso_queso_total,sal_queso
fecha_index,,,,,,,,,,,,,,,,,,
2022-03-01,01/03/2022,vaca,14910,14.00,5.27,4.17,15810.99,12.85,5.03,3.97,3.85,6.38,barra,55.94,30.50,5.41,2351,1.55
2022-03-04,04/03/2022,vaca,14100,13.60,5.11,4.15,14989.74,12.50,4.87,3.95,3.68,6.41,barra,54.58,29.80,5.88,2266,1.77
2022-03-25,25/03/2022,vaca,13310,13.51,5.07,4.04,13011.79,12.37,4.83,3.84,3.70,6.42,barra,54.10,30.63,5.48,1867,1.32
2022-04-01,01/04/2022,vaca,15430,13.50,5.16,4.04,15946.80,12.60,4.92,3.84,3.84,6.48,barra,56.76,30.51,5.36,2261,1.36
2022-04-08,08/04/2022,vaca,15930,13.71,5.09,4.05,16452.76,12.67,4.85,3.85,3.97,6.41,barra,55.63,30.01,5.31,2346,1.90


Los datos se corresponden con un conjunto de fabricaciones de queso de vaca, como hemos dicho. LAs principales variables recogidas son las siguientes:

In [20]:
# Diccionario de variables con espacio para completar descripciones
descripcion_variables = {
    'fecha': 'Fecha de producción',
    'receta': 'Tipo de receta utilizada',
    'litros_past': 'Litros de leche pasteurizada',
    'est_past': 'Extracto seco total de la leche pasteurizada (g/100 ml leche)',
    'mg_past': 'Materia grasa de la leche pasteurizada (g/100 ml leche)',
    'mp_past': 'Materia proteica de la leche pasteurizada (g/100 ml leche)',
    'kg_cuba': 'Kilos totales de leche en la cuba',
    'est_cuba': 'Extracto seco total en cuba (g/100 g leche)',
    'mg_cuba': 'Materia grasa en cuba (g/100 g leche)',
    'mp_cuba': 'Materia proteica en cuba (g/100 g leche)',
    'lactosa_cuba': 'Contenido de lactosa en cuba (g/100 g leche)',
    'ph_final_moldeo_cuba': 'pH final antes del moldeo',
    'formato': 'Formato del queso una vez moldeado',
    'est_salida_salmuera': 'Extracto seco total tras salida de salado (g/100 g queso)',
    'mg_salida_salmuera': 'Materia grasa tras salida de salado (g/100 g queso)',
    'ph_salida_salmuera': 'pH tras salida de salado',
    'peso_queso_total': 'Peso total del queso producido (medido a la salida de la salmuera)',
    'sal_queso': 'Contenido de sal en el queso tras salida de salado (g/ 100 g queso)'
}

# Imprimir como tabla Markdown
# print("| Variable             | Descripción                                                            |")
# print("|----------------------|------------------------------------------------------------------------|")
# for variable, descripcion in descripcion_variables.items():
#     print(f"| {variable:<20} | {descripcion:<70} |")

# Construir tabla HTML
html = """
<table border="1">
    <thead>
        <tr>
           <th style="text-align:left;"><strong>Variable</strong></th>
           <th style="text-align:left;"><strong>Descripción</strong></th>
        </tr>
    </thead>
    <tbody>
"""

for variable, descripcion in descripcion_variables.items():
    html += f'        <tr><td style="text-align:left;">{variable}</td><td style="text-align:left;">{descripcion}</td></tr>\n'

html += "    </tbody>\n</table>"

# Mostrar en entorno compatible (como Jupyter Notebook)
from IPython.display import display, HTML
display(HTML(html))



Variable,Descripción
fecha,Fecha de producción
receta,Tipo de receta utilizada
litros_past,Litros de leche pasteurizada
est_past,Extracto seco total de la leche pasteurizada (g/100 ml leche)
mg_past,Materia grasa de la leche pasteurizada (g/100 ml leche)
mp_past,Materia proteica de la leche pasteurizada (g/100 ml leche)
kg_cuba,Kilos totales de leche en la cuba
est_cuba,Extracto seco total en cuba (g/100 g leche)
mg_cuba,Materia grasa en cuba (g/100 g leche)
mp_cuba,Materia proteica en cuba (g/100 g leche)


El laboratorio nos facillita la información analítica de la leche pasteurizada en porcentaje masa/volumen, mientas que la leche en cubas y el queso es masa/masa. Esto es así porque la cantidad de leche que pasa por el pasteurizador se mide con un contador volumétrico, mientras que en la cuba se mide con una célula de carga.

Si el volumen de leche en cuba se midiese también con contador volumétrico, necesitaríamos convertir los resultados analíticos a masa/volumen para obtener los kilos de cada materia, utilizando la densidad.

En la práctica, usar una densidad promedio fija o ajustada por estaciones funciona bastante bien, y evita los errores de medida de este parámetro que parecen sencillos pero siempre están sometidos al error de muestreo, además de la necesidad de corregir la temperatura.



Veamos los datos analíticos del queso

In [14]:
# Calcular media y mediana
media_est = df['est_salida_salmuera'].mean()
mediana_est = df['est_salida_salmuera'].median()

media_mg = df['mg_salida_salmuera'].mean()
mediana_mg = df['mg_salida_salmuera'].median()

# Crear tabla HTML
html = f"""
<table border="1">
    <thead>
        <tr>
            <th><strong>Composición del queso</strong></th>
            <th><strong>Media (g/100 g)</strong></th>
            <th><strong>Mediana (g/100 g)</strong></th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>EST a salida de salado</td>
            <td>{media_est:.2f}%</td>
            <td>{mediana_est:.2f}%</td>
        </tr>
        <tr>
            <td>MG a salida de salado</td>
            <td>{media_mg:.2f}%</td>
            <td>{mediana_mg:.2f}%</td>
        </tr>
    </tbody>
</table>
"""

# Mostrar tabla en notebook
from IPython.display import display, HTML
display(HTML(html))

Composición del queso,Media (g/100 g),Mediana (g/100 g)
EST a salida de salado,55.27%,55.63%
MG a salida de salado,30.15%,30.05%


Para hacer este producto, estamos trabajando con una leche cuya composición es:

In [21]:
# Calcular medias y medianas desde df
media_est = df['est_cuba'].mean()
mediana_est = df['est_cuba'].median()

media_mg = df['mg_cuba'].mean()
mediana_mg = df['mg_cuba'].median()

media_mp = df['mp_cuba'].mean()
mediana_mp = df['mp_cuba'].median()

# Crear tabla HTML
html = f"""
<table border="1">
    <thead>
        <tr>
            <th style="text-align:left;"><strong>Composición materia prima</strong></th>
            <th style="text-align:left;"><strong>Media (g/L)</strong></th>
            <th style="text-align:left;"><strong>Mediana (g/L)</strong></th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="text-align:left;">Extracto seco total (ESM)</td>
            <td style="text-align:left;">{media_est:.2f}</td>
            <td style="text-align:left;">{mediana_est:.2f}</td>
        </tr>
        <tr>
            <td style="text-align:left;">Materia grasa (MG)</td>
            <td style="text-align:left;">{media_mg:.2f}</td>
            <td style="text-align:left;">{mediana_mg:.2f}</td>
        </tr>
        <tr>
            <td style="text-align:left;">Proteínas totales (MP)</td>
            <td style="text-align:left;">{media_mp:.2f}</td>
            <td style="text-align:left;">{mediana_mp:.2f}</td>
        </tr>
    </tbody>
</table>
"""

# Mostrar en entorno compatible (como Jupyter Notebook)
from IPython.display import display, HTML
display(HTML(html))

Composición materia prima,Media (g/L),Mediana (g/L)
Extracto seco total (ESM),12.63,12.64
Materia grasa (MG),4.87,4.87
Proteínas totales (MP),3.91,3.92


## Cálculo de la cantidad de leche necesaria para fabricar 1 kg de queso

Sabemos que nuestro proceso de fabricación quesera consiste en la coagulación de la caseína y expulsión de suero, y que la materia grasa queda retenida en la red de caseína.

Este concepto es la base fundamental de la comprensión de la tecnología quesera: es el rendimiento en la recuperación de la proteína el que nos va a definir todo el proceso, la materia grasa “acompañará” de forma estática (aunque tendrá influencia en el desuerado)

Por esta razón, el principal elemento del rendimiento en la definición de la tecnología es el porcentaje de recuperación de proteína en el extracto seco magro (aislamos el efecto la grasa)

Hemos definido nuestro producto mediante los análisis de EST y MG; necesitamos saber el resto de parámetros, en concreto el ESM que estamos definiendo para nuestro producto:


In [24]:
# Calcular medias y medianas
media_est = df['est_salida_salmuera'].mean()
mediana_est = df['est_salida_salmuera'].median()

media_mg = df['mg_salida_salmuera'].mean()
mediana_mg = df['mg_salida_salmuera'].median()

# Calcular ESM (Extracto Seco Magro)
media_esm = media_est - media_mg
mediana_esm = mediana_est - mediana_mg

# Calcular índice MG/EST correctamente
media_mg_est = (media_mg / media_est) * 100
mediana_mg_est = (mediana_mg / mediana_est) * 100

# Crear tabla HTML
html = f"""
<table border="1">
    <thead>
        <tr>
            <th style="text-align:left;"><strong>Composición del queso</strong></th>
            <th style="text-align:left;"><strong>Media (g/100 g)</strong></th>
            <th style="text-align:left;"><strong>Mediana (g/100 g)</strong></th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="text-align:left;">EST a salida de salado</td>
            <td style="text-align:left;">{media_est:.2f}%</td>
            <td style="text-align:left;">{mediana_est:.2f}%</td>
        </tr>
        <tr>
            <td style="text-align:left;">MG a salida de salado</td>
            <td style="text-align:left;">{media_mg:.2f}%</td>
            <td style="text-align:left;">{mediana_mg:.2f}%</td>
        </tr>
        <tr>
            <td style="text-align:left;">ESM a salida de salado)</td>
            <td style="text-align:left;">{media_esm:.2f}%</td>
            <td style="text-align:left;">{mediana_esm:.2f}%</td>
        </tr>
        <tr>
            <td style="text-align:left;">MG/EST a salida de salado</td>
            <td style="text-align:left;">{media_mg_est:.2f}%</td>
            <td style="text-align:left;">{mediana_mg_est:.2f}%</td>
        </tr>
    </tbody>
</table>
"""

# Mostrar tabla en notebook
from IPython.display import display, HTML
display(HTML(html))

Composición del queso,Media (g/100 g),Mediana (g/100 g)
EST a salida de salado,55.27%,55.63%
MG a salida de salado,30.15%,30.05%
ESM a salida de salado),25.13%,25.58%
MG/EST a salida de salado,54.54%,54.02%


Ahora necesitamos conocer cuál es la tasa o ratio de conversión en ESM de queso del ESM de la leche en cubas. Para ello calculamos este ratio, que vamos a llamar **coeficiente de recuperación de ESM**.